In [ ]:
pip install pymysql

In [99]:
import pymysql
import os 
import datetime as dt

In [ ]:
# print(dir(os))

In [101]:
# To get the Password 
# Method 1: os.getenv("varibale_name", "default value")
# Method 2: environ[] or environ.get()
PASSWORD = os.getenv("MYSQL_PASSWORD", "1234") # Method 1
PASSWORD1 = os.environ["MYSQL_PASSWORD"] # Method 2

if PASSWORD:
    print("Password was gotten.")

Password was gotten.


In [103]:
# To create a connection between Mysql And Python 
details = {"user":"root","port":3306, "host":"localhost", "password" : PASSWORD}
try: 
    db = pymysql.connect(**details)
    cursor = db.cursor()
    if cursor:
        print("Connection successful✅")
except Exception as e:
    print(f"Error Message: {e}.")

Connection successful✅


In [7]:
# To show all databases
cursor.execute("Show databases")
for i in cursor:
    print(i)

('airline',)
('ars',)
('bank',)
('car_dealer',)
('day12',)
('db15',)
('earlycode',)
('ec_mfb',)
('eedc_db',)
('glovo',)
('hospital',)
('hotel',)
('inec',)
('information_schema',)
('movie_rental',)
('mtn_db',)
('mysql',)
('performance_schema',)
('pop_sys',)
('portal',)
('sakila',)
('school_db',)
('shoe_collections',)
('sport',)
('sys',)
('world',)


In [9]:
# To show all tables on the Ars database 
cursor.execute("use ars")
cursor.execute("show tables in ars")
for i in cursor:
    print(i)

('booking',)
('train',)


In [105]:
import string as st 
print(dir(st))

['Formatter', 'Template', '_ChainMap', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_re', '_sentinel_dict', '_string', 'ascii_letters', 'ascii_lowercase', 'ascii_uppercase', 'capwords', 'digits', 'hexdigits', 'octdigits', 'printable', 'punctuation', 'whitespace']


In [107]:
# Function to generate booking_id
def generate_bid():
    import string as st
    import random
    character = st.ascii_uppercase + st.digits
    booking_id = "".join(random.choice(character) for _ in range(11))
    return booking_id
generate_bid()

'Q8J9QN36K7R'

In [21]:
cursor.execute("describe train")
for i in cursor:
    print(i)

('train_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('train_station', 'varchar(100)', 'NO', '', None, '')
('location', 'varchar(100)', 'NO', '', None, '')
('Destination', 'varchar(100)', 'NO', '', None, '')
('Total_seats', 'int', 'NO', '', None, '')
('Available_seat', 'int', 'NO', '', None, '')
('Fare', 'decimal(10,2)', 'YES', '', None, '')
('Time_departure', 'time', 'YES', '', None, '')


In [155]:
cursor.execute("use ars")
# To create a Function that stores all train records
def train_records():
    print("\n" + "=" * 40)
    print("\tTrain Records")
    print("=" * 40)
    try:
        train_station = input("Enter Train station: ")
        location = input("Enter the Current Location: ")
        Destination = input("Enter the Destination: ")
        Total_seats = int(input("Enter the Total seats: "))
        Available_seat = int(input("Enter the available seats for this trip: "))
        Fare = float(input("Enter the Fare for this trip: ₦"))
        time_departure = input("Enter the time for departure(H:M): ")
        query = """INSERT INTO TRAIN(train_station, location, Destination, Total_seats, Available_seat, Fare, Time_departure)
        values(%s, %s, %s, %s, %s, %s, %s)"""
        if Available_seat != Total_seats:
            print('wrong upload')
            db.rollback()
        cursor.execute(query, (train_station, location, Destination, Total_seats, Available_seat, Fare, 
                               dt.datetime.strptime(time_departure, "%H:%M").time()))
        db.commit()
        print("Train Uploads was successful✅..")
    except Exception as e:
        print(f"Error Message:{e}")
        db.rollback()

def display_trips():
    try:
        cursor.execute("select * from train")
        result = cursor.fetchall()
        if not result:
            print("No up coming trips.")
        else:
            for i in result:
                print(f"\nTrain Id:{i[0]}|Station: {i[1]} |Location: {i[2]} |Destination: {i[3]} |Total seats: {i[4]} |Available seats: {i[5]} |Fare: ₦{i[6]}|Time Departure: {i[7]}")
    except Exception as e:
        print(f"Error message: {e}")
        db.rollback()
        
def booking():
    print("\n" + "=" * 40)
    print("\tBooking Section")
    print("=" * 40)
    try:
        passanger_name = input("Enter your name: ")
        seat_to_book = int(input("How many seats do you want to book: "))
        Gender = input("Enter your Gender(Male or Female): ")
        train_id = int(input("Enter the Train ID: "))
        booking_id = generate_bid()
        booking_status = "Confirmed"
    
        # To check if the train exists.
        query = "Select Fare, Available_seat from train where train_id = %s"
        cursor.execute(query, train_id)
        output = cursor.fetchone()
        if not output:
            print("Trip does not exist.")
        fare, Available_seat = output
        
        if Available_seat < seat_to_book:
            print("No more seat")
        total_cost = fare * seat_to_book
    
        # To update the booking table and inform the user it was successful.
        query = "INSERT INTO BOOKING(Booking_id, Customer_name, Seat_to_book, train_id, Gender, booking_status) values(%s, %s, %s, %s, %s, %s)"
        cursor.execute(query, (booking_id, passanger_name, seat_to_book, train_id, Gender, booking_status))
        

        # Update the Train table.
        cursor.execute("Update train set Available_seat = Available_seat - %s where train_id %s", (Available_seat, train_id))
        db.commit()
        print(f"Congratulation, Your booking id is{booking_id}. Please take this Amount ₦{total_cost} to the bank to pay.")
    except Exception as e:
        print(f"Error message:{e}")
        db.rollback()
        
def confirm_trip():
    print("\n" + "=" * 40)
    print("\tTO Confirm Trip")
    print("=" * 40)
    try:
        booking_id = input("Enter your booking id: ")
        cursor.execute("""Select a.train_id, a.destination, a.fare, a.time_departure, b.customer_name, b.Gender from
         train as a join booking as b a.train = b.train_id""")
        result = cursor.fetchone()
        if not result:
            print("No records")
        for i in result:
            print(i)
    except Exception as e:
        print("Error Message", e)
        
def main():
    while True:
        try: 
            print("\n" + "=" * 40)
            print("\tWelcome to Abuja Train Terminal")
            print("=" * 40)
            print("1. >> Upload Upcoming Trips.\n2. >> Upcoming Trip\n3. >> Booking Trip\n4. >> Confirm Trip\n5. >> logout")
            choice = int(input("kindly Make a selection from the option above: "))
            if choice == 1:
                train_records()
            elif choice == 2:
                display_trips()
            elif choice == 3:
                booking()
            elif choice == 4:
                confirm_trips()
            elif choice == 5:
                print("Thank you using Abuja Train Terminal..")
                break
            else:
                print("Wrong selection, Please try again")
        except Exception as e:
            print("Error Message: ",e)

In [ ]:
main()


	Welcome to Abuja Train Terminal
1. >> Upload Upcoming Trips.
2. >> Upcoming Trip
3. >> Booking Trip
4. >> Confirm Trip
5. >> logout


kindly Make a selection from the option above:  1



	Train Records


Enter Train station:  Udo Station 
Enter the Current Location:  Abuja 
Enter the Destination:  Lagos
Enter the Total seats:  200
Enter the available seats for this trip:  120
Enter the Fare for this trip: ₦ 28950
Enter the time for departure(H:M):  7:30


wrong upload
Train Uploads was successful✅..

	Welcome to Abuja Train Terminal
1. >> Upload Upcoming Trips.
2. >> Upcoming Trip
3. >> Booking Trip
4. >> Confirm Trip
5. >> logout


kindly Make a selection from the option above:  2



Train Id:1|Station: Kubwa |Location: Abuja |Destination: Kaduna |Total seats: 300 |Available seats: 300 |Fare: ₦5000.00|Time Departure: 7:30:00

Train Id:2|Station: Kubwa Station  |Location: Abuja |Destination: Kaduna |Total seats: 30 |Available seats: 30 |Fare: ₦5000.00|Time Departure: 7:00:00

Train Id:3|Station: Udo Station  |Location: Abuja  |Destination: Lagos |Total seats: 200 |Available seats: 120 |Fare: ₦28950.00|Time Departure: 7:30:00

	Welcome to Abuja Train Terminal
1. >> Upload Upcoming Trips.
2. >> Upcoming Trip
3. >> Booking Trip
4. >> Confirm Trip
5. >> logout


In [ ]:
try:
    cursor.execute("alter table booking drop index Seat_number")
    cursor.execute("alter table booking rename column seat_number to seat_to_book")
    if cursor:
        print("Successful")
except Exception as e:
    print(e)

In [113]:
cursor.execute("describe train")
for i in cursor:
    print(i)

('train_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('train_station', 'varchar(100)', 'NO', '', None, '')
('location', 'varchar(100)', 'NO', '', None, '')
('Destination', 'varchar(100)', 'NO', '', None, '')
('Total_seats', 'int', 'NO', '', None, '')
('Available_seat', 'int', 'NO', '', None, '')
('Fare', 'decimal(10,2)', 'YES', '', None, '')
('Time_departure', 'time', 'YES', '', None, '')


In [ ]:
# pip install numpy pandas plotly matplotlib nbformat 